# GW170817: Posterior plot comparison

In [1]:
import matplotlib.pyplot as plt
%matplotlib inline
import bilby
import pandas as pd
import numpy as np
import seaborn as sns
from bilby.gw.result import CBCResult
import matplotlib.lines as mlines
from kde_contour import Bounded_1d_kde, kdeplot_2d_clevels

### Waveforms: TaylorF2Ecck, TaylorF2Ecc 3PN

In [10]:
rng = np.random.default_rng(12345)

sns.set_theme(palette='colorblind', font_scale=1.5)

result1 = bilby.result.read_in_result("../result_files/TF2Ecck_3PN_bayeswave.hdf5").posterior
result2 = bilby.result.read_in_result("../result_files/TF2Ecc_3PN_bayeswave.hdf5").posterior
nsamples = min(len(result1), len(result2))

result_ecck = result1.sample(nsamples, random_state=rng)
result_ecck["templates"] = np.full(len(result_ecck), "Ecck 3PN")
result_ecck["chirp_mass"] = result_ecck["chirp_mass"]
result_ecck["eccentricity"] = result_ecck["eccentricity"]


result_ecc3PN = result2.sample(nsamples, random_state=rng)
result_ecc3PN["templates"] = np.full(len(result_ecc3PN), "Ecc 3PN")
result_ecc3PN["chirp_mass"] = result_ecc3PN["chirp_mass"]
result_ecc3PN["eccentricity"] = result_ecc3PN["eccentricity"]

result = pd.concat([result_ecck, result_ecc3PN],ignore_index=True)

In [14]:
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

plt.rcParams.update({
    'backend': 'Agg',
    'savefig.dpi': 300,
    'grid.alpha': 0.5,
    'path.simplify': True,
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    'mathtext.fontset': 'custom',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.minor.width': 1,
    'ytick.minor.width': 1,
    'lines.markeredgewidth': 1,
    'legend.numpoints': 1,
    'legend.frameon': False,
    'legend.handletextpad': 0.3
})

lw = 1

def kdeplot2d(x, y, rng=12345, **kws):
    kws.pop('label', None)
    kdeplot_2d_clevels(xs=x, ys=y, auto_bound=True, linewidths=lw, rng=rng, **kws)

def kdeplot1d(x, **kws):
    if np.all(x.isna()):
        return
    for key in ['label', 'hue_order', 'color']:
        kws.pop(key, None)
    df = pd.DataFrame({'x': x, 'y': Bounded_1d_kde(x, xlow=min(x)-min(x)*0.1, xhigh=max(x)+max(x)*0.1, **kws)(x)})
    df = df.sort_values(['x'])
    plt.fill_between(df['x'], df['y'], np.zeros(len(x)), alpha=0.1)
    plt.plot(df['x'], df['y'], lw=lw)
    plt.xlim(df['x'].min()-df['x'].min()*0.1, df['x'].max()+df['x'].max()*0.1)
    current_ymax = plt.ylim()[1]
    if current_ymax > df['y'].max()*1.05:
        plt.ylim(0,current_ymax)
    else:
        plt.ylim(0,df['y'].max()*1.05)

vars = ['chirp_mass', 'mass_ratio', 'eccentricity']
g = sns.PairGrid(data=result,
                 vars=vars,
                 corner=True, hue="templates",
                 diag_sharey=False,
                 layout_pad=0.,
                 height=1.5,
                )
g.map_lower(kdeplot2d, levels=[0.864,0.393])
g.map_diag(kdeplot1d)
# 32s

In [23]:
g.axes[2, 2].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 1].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 0].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 0].tick_params(axis='y', rotation=45, pad=1)
g.axes[1, 0].tick_params(axis='y', rotation=45, pad=1)
# set maximum number of ticks on each axis
g.axes[2, 2].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 1].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 0].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 0].yaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[1, 0].yaxis.set_major_locator(plt.MaxNLocator(3))

g.axes[2, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.5f' % x))
g.axes[2, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.2f' % x))
g.axes[2, 2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.3f' % x))
g.axes[2, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.3f' % x))
g.axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.2f' % x))      

g.axes[2,0].set_xlabel(r'$\mathcal{M}$', labelpad=2, fontsize=12)
g.axes[1,0].set_ylabel(r'$q$', labelpad=8, fontsize=12)
g.axes[2,1].set_xlabel(r'$q$', labelpad=15, fontsize=12)
g.axes[2,0].set_ylabel(r'$e_0$', labelpad=4, fontsize=12)
g.axes[2,2].set_xlabel(r'$e_0$', labelpad=11, fontsize=12)

min_mc = 1.19685
max_mc = 1.19760
min_q = 0.60
max_q = 0.81
min_e0 = 0.001
max_e0 = 0.0215
g.axes[2,0].set_xlim(min_mc, max_mc)
g.axes[2,0].set_ylim(min_e0, max_e0)
g.axes[2,2].set_xlim(min_e0, max_e0)
g.axes[2,1].set_xlim(min_q, max_q)
g.axes[1,0].set_ylim(min_q, max_q)

# add grid
for i in range(3):
    for j in range(3):
        if g.axes[i, j] is not None:
            g.axes[i, j].grid(True, which="both", ls="-", alpha=0.5)

# add legend with line colors
blue_line = mlines.Line2D([], [], color='C0', label='Ecck 3PN')
orange_line = mlines.Line2D([], [], color='C1', label='Ecc 3PN')
handles = [blue_line, orange_line]
labels = [h.get_label() for h in handles] 
g.fig.legend(handles=handles, labels=labels, bbox_to_anchor=(0., 0.9, 0.9, .0), ncol=1) # Adjust loc and ncol as needed

g.savefig("GW170817a.pdf", bbox_inches="tight", dpi=300)
plt.show()

### Waveforms: TaylorF2Ecck, TaylorF2Ecc 3PN, TaylorF2Ecc 3.5PN

In [24]:
rng = np.random.default_rng(12345)

sns.set_theme(palette='colorblind', font_scale=1.5)

result4 = bilby.result.read_in_result("../result_files/TF2Ecc_3p5PN_bayeswave.hdf5").posterior
result_ecc3p5nospin = result4.sample(nsamples, random_state=rng)
result_ecc3p5nospin["templates"] = np.full(len(result_ecc3p5nospin), "Ecc 3.5PN")
result_ecc3p5nospin["chirp_mass"] = result_ecc3p5nospin["chirp_mass"]
result_ecc3p5nospin["eccentricity"] = result_ecc3p5nospin["eccentricity"]

result = pd.concat([result_ecck, result_ecc3p5nospin],ignore_index=True)

In [25]:
import matplotlib as mpl
mpl.rcParams.update(mpl.rcParamsDefault)

plt.rcParams.update({
    'backend': 'Agg',
    'savefig.dpi': 300,
    'grid.alpha': 0.5,
    'path.simplify': True,
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    'mathtext.fontset': 'custom',
    'xtick.major.size': 6,
    'ytick.major.size': 6,
    'xtick.minor.size': 3,
    'ytick.minor.size': 3,
    'xtick.major.width': 1,
    'ytick.major.width': 1,
    'xtick.minor.width': 1,
    'ytick.minor.width': 1,
    'lines.markeredgewidth': 1,
    'legend.numpoints': 1,
    'legend.frameon': False,
    'legend.handletextpad': 0.3
})

lw = 1

def kdeplot2d(x, y, rng=12345, **kws):
    kws.pop('label', None)
    kdeplot_2d_clevels(xs=x, ys=y, auto_bound=True, linewidths=lw, rng=rng, **kws)

vars = ['chirp_mass', 'mass_ratio', 'eccentricity']
g = sns.PairGrid(data=result,
                 vars=vars,
                 corner=True, hue="templates",
                 palette=['C0', 'C3'],
                 diag_sharey=False,
                 layout_pad=0.,
                 height=1.5,
                )
g.map_lower(kdeplot2d, levels=[0.864,0.393])
# 32s

In [26]:
def kdeplot1d(x, **kws):
    #  define colors for plt
    if kws['label']=='Ecck 3PN':
        color = 'C0'
    elif kws['label']=='Ecc 3.5PN':
        color = 'C3'
    else:
        raise ValueError('Invalid template')
    
    if np.all(x.isna()):
        return
    for key in ['label', 'hue_order', 'color', 'ax']:
        kws.pop(key, None)
    df = pd.DataFrame({'x': x, 'y': Bounded_1d_kde(x, xlow=min(x), xhigh=max(x), **kws)(x)})
    df = df.sort_values(['x'])
    plt.fill_between(df['x'], df['y'], np.zeros(len(x)), alpha=0.1, color=color)
    plt.plot(df['x'], df['y'], lw=lw, color=color)
    plt.xlim(df['x'].min(), df['x'].max())
    current_ymax = plt.ylim()[1]
    if current_ymax > df['y'].max()*1.05:
        plt.ylim(0,current_ymax)
    else:
        current_ymax = df['y'].max()*1.05
        plt.ylim(0,df['y'].max()*1.05)
    print(current_ymax)

g.map_diag(kdeplot1d)

3764.237523622598
5154.321472158059
13.925949259035274
13.925949259035274
72.20365754956052
96.14266179472823


In [34]:
g.axes[2, 2].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 1].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 0].tick_params(axis='x', rotation=45, pad=1)
g.axes[2, 0].tick_params(axis='y', rotation=45, pad=1)
g.axes[1, 0].tick_params(axis='y', rotation=45, pad=1)

g.axes[2, 2].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 1].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 0].xaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[2, 0].yaxis.set_major_locator(plt.MaxNLocator(3))
g.axes[1, 0].yaxis.set_major_locator(plt.MaxNLocator(3))

g.axes[2, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.5f' % x))
g.axes[2, 1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.2f' % x))
g.axes[2, 2].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.3f' % x))
g.axes[2, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.3f' % x))
g.axes[1, 0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '%.2f' % x))      

g.axes[2,0].set_xlabel(r'$\mathcal{M}$', labelpad=2, fontsize=12)
g.axes[1,0].set_ylabel(r'$q$', labelpad=8, fontsize=12)
g.axes[2,1].set_xlabel(r'$q$', labelpad=15, fontsize=12)
g.axes[2,0].set_ylabel(r'$e_0$', labelpad=4, fontsize=12)
g.axes[2,2].set_xlabel(r'$e_0$', labelpad=11, fontsize=12)

min_mc = 1.19685
max_mc = 1.19765
min_q = 0.60
max_q = 0.95
min_e0 = 0.001
max_e0 = 0.0215
g.axes[2,0].set_xlim(min_mc, max_mc)
g.axes[2,0].set_ylim(min_e0, max_e0)
g.axes[2,2].set_xlim(min_e0, max_e0)
g.axes[2,1].set_xlim(min_q, max_q)
g.axes[1,0].set_ylim(min_q, max_q)

# add grid
for i in range(3):
    for j in range(3):
        if g.axes[i, j] is not None:
            g.axes[i, j].grid(True, which="both", ls="-", alpha=0.5)

# add legend with line colors
blue_line = mlines.Line2D([], [], color='C0', label='Ecck 3PN')
green_line = mlines.Line2D([], [], color='C3', label='Ecc 3.5PN')
handles = [blue_line, green_line]
labels = [h.get_label() for h in handles] 
g.fig.legend(handles=handles, labels=labels, bbox_to_anchor=(0., 0.9, 0.9, .0), ncol=1) # Adjust loc and ncol as needed

g.savefig("GW170817c.pdf", bbox_inches="tight", dpi=300)
plt.show()